In [0]:
# CELL 1 — Load the scored data
spark.sql("USE pothole_heatmap")

df = spark.sql("""
    SELECT
        texture_depth_variance,
        bump_intensity,
        crack_intensity,
        rut_depth,
        edge_roughness,
        longitudinal_variance,
        accidents_per_km,
        daily_traffic,
        CASE road_type
            WHEN 'highway'   THEN 3
            WHEN 'arterial'  THEN 2
            WHEN 'collector' THEN 1
            ELSE 0
        END as road_type_encoded,
        pothole_detected as label
    FROM pothole_heatmap.road_sections_raw
""")

print(f"✅ Loaded {df.count():,} rows")
print(f"   Potholes (1): {df.filter('label=1').count():,}")
print(f"   No pothole (0): {df.filter('label=0').count():,}")
df.show(5)

✅ Loaded 50,000 rows
   Potholes (1): 24,871
   No pothole (0): 25,129
+----------------------+--------------+---------------+---------+--------------+---------------------+----------------+-------------+-----------------+-----+
|texture_depth_variance|bump_intensity|crack_intensity|rut_depth|edge_roughness|longitudinal_variance|accidents_per_km|daily_traffic|road_type_encoded|label|
+----------------------+--------------+---------------+---------+--------------+---------------------+----------------+-------------+-----------------+-----+
|                  2.83|          3.91|           0.28|     5.45|           2.3|                  8.9|            7.19|        62403|                0|    1|
|                  9.83|           5.0|           3.72|     3.75|          7.77|                 4.22|            1.69|        57344|                3|    0|
|                   9.4|          0.81|           8.27|     9.76|          0.47|                 7.54|            3.11|         3164|      

In [0]:
# CELL 2 — Prepare features for Spark ML
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

feature_cols = [
    "texture_depth_variance",
    "bump_intensity",
    "crack_intensity",
    "rut_depth",
    "edge_roughness",
    "longitudinal_variance",
    "accidents_per_km",
    "daily_traffic",
    "road_type_encoded"
]

# VectorAssembler combines all features into one column called "features"
# This is required by Spark ML — it's like X in sklearn
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

data = assembler.transform(df).select("features", "label")

# Train/test split — 80% train, 20% test
train_data, test_data = data.randomSplit([0.8, 0.2], seed=42)

print(f"✅ Training set: {train_data.count():,} rows")
print(f"   Test set:     {test_data.count():,} rows")

✅ Training set: 39,948 rows
   Test set:     10,052 rows


In [0]:
# CELL 3 — Train Random Forest
from pyspark.ml.classification import RandomForestClassifier

rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,        # 100 decision trees in the forest
    maxDepth=10,         # each tree can be 10 levels deep
    seed=42
)

print("Training Random Forest... (this takes ~1-2 mins)")
rf_model = rf.fit(train_data)
print("✅ Random Forest trained!")
print(f"   Number of trees: {rf_model.getNumTrees}")

Training Random Forest... (this takes ~1-2 mins)
✅ Random Forest trained!
   Number of trees: 100


In [0]:
# CELL 4 — Evaluate the model
predictions = rf_model.transform(test_data)

# AUC-ROC score (main metric for imbalanced datasets)
auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = auc_evaluator.evaluate(predictions)

# Accuracy
acc_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)
accuracy = acc_evaluator.evaluate(predictions)

# Recall on positive class (how many potholes did we catch?)
recall_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="weightedRecall"
)
recall = recall_evaluator.evaluate(predictions)

print("=" * 40)
print("RANDOM FOREST RESULTS")
print("=" * 40)
print(f"AUC-ROC:  {auc:.3f}  (1.0 = perfect, 0.5 = random)")
print(f"Accuracy: {accuracy:.3f}")
print(f"Recall:   {recall:.3f}")
print("=" * 40)

# Compare with research paper
print("\nComparison with Abed et al. (2023) paper:")
print(f"  Paper accuracy on pothole sections: 55.5%")
print(f"  Our RF accuracy: {accuracy*100:.1f}%")

RANDOM FOREST RESULTS
AUC-ROC:  0.639  (1.0 = perfect, 0.5 = random)
Accuracy: 0.598
Recall:   0.598

Comparison with Abed et al. (2023) paper:
  Paper accuracy on pothole sections: 55.5%
  Our RF accuracy: 59.8%


In [0]:
# CELL 5 — Feature importance
# Which road condition matters most for predicting potholes?
import pandas as pd

importances = rf_model.featureImportances.toArray()
feature_importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': importances
}).sort_values('importance', ascending=False)

print("Feature Importance (what matters most for pothole prediction):")
print("=" * 55)
for _, row in feature_importance_df.iterrows():
    bar = "█" * int(row['importance'] * 200)
    print(f"  {row['feature']:30s} {row['importance']:.4f}  {bar}")

Feature Importance (what matters most for pothole prediction):
  crack_intensity                0.2203  ████████████████████████████████████████████
  texture_depth_variance         0.1620  ████████████████████████████████
  rut_depth                      0.1402  ████████████████████████████
  bump_intensity                 0.1185  ███████████████████████
  edge_roughness                 0.0903  ██████████████████
  accidents_per_km               0.0819  ████████████████
  daily_traffic                  0.0802  ████████████████
  longitudinal_variance          0.0798  ███████████████
  road_type_encoded              0.0268  █████


In [0]:
# CELL 6 — Save RF predictions to Delta Lake
predictions_to_save = predictions.select(
    "features",
    "label",
    "prediction",
    "probability"
).withColumnRenamed("label", "actual") \
 .withColumnRenamed("prediction", "rf_predicted")

predictions_to_save.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("pothole_heatmap.rf_predictions")

print("✅ RF predictions saved to Delta Lake")
print(f"   Table: pothole_heatmap.rf_predictions")

✅ RF predictions saved to Delta Lake
   Table: pothole_heatmap.rf_predictions


In [0]:
# CELL 7 — Fix class imbalance using oversampling
# Then retrain and compare

from pyspark.sql import functions as F

# Separate minority and majority class
pothole_df    = train_data.filter("label = 1")
no_pothole_df = train_data.filter("label = 0")

pothole_count    = pothole_df.count()
no_pothole_count = no_pothole_df.count()
ratio = no_pothole_count // pothole_count

print(f"Before balancing:")
print(f"  Pothole sections:    {pothole_count:,}")
print(f"  No pothole sections: {no_pothole_count:,}")
print(f"  Imbalance ratio:     {ratio}:1")

# Oversample minority class (repeat pothole examples)
oversampled_pothole = pothole_df.sample(
    withReplacement=True, 
    fraction=float(ratio), 
    seed=42
)

# Combine balanced dataset
balanced_train = no_pothole_df.union(oversampled_pothole)

print(f"\nAfter balancing:")
print(f"  Pothole sections:    {balanced_train.filter('label=1').count():,}")
print(f"  No pothole sections: {balanced_train.filter('label=0').count():,}")

Before balancing:
  Pothole sections:    19,894
  No pothole sections: 20,054
  Imbalance ratio:     1:1

After balancing:
  Pothole sections:    20,101
  No pothole sections: 20,054


In [0]:
# CELL 8 — Retrain on balanced data
rf_balanced = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    numTrees=100,
    maxDepth=10,
    seed=42
)

print("Training on balanced data...")
rf_balanced_model = rf_balanced.fit(balanced_train)

# Evaluate
predictions_balanced = rf_balanced_model.transform(test_data)

auc_balanced = auc_evaluator.evaluate(predictions_balanced)
acc_balanced = acc_evaluator.evaluate(predictions_balanced)

print("\n" + "=" * 45)
print("COMPARISON: Before vs After Balancing")
print("=" * 45)
print(f"{'Metric':<12} {'Before':>10} {'After':>10}")
print("-" * 45)
print(f"{'AUC-ROC':<12} {auc:>10.3f} {auc_balanced:>10.3f}")
print(f"{'Accuracy':<12} {accuracy:>10.3f} {acc_balanced:>10.3f}")
print("=" * 45)
print("\nAUC closer to 1.0 = better at finding actual potholes")

Training on balanced data...

COMPARISON: Before vs After Balancing
Metric           Before      After
---------------------------------------------
AUC-ROC           0.639      0.641
Accuracy          0.598      0.605

AUC closer to 1.0 = better at finding actual potholes
